In [41]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Testing CCA on benchmark data

This notebook aims to run my implementation of CCA on benchmark data and compare the results with literature

In [42]:
from ssvep.datasets import load_benchmark
from ssvep import config

X, y, blocks = load_benchmark(1)

print(X.shape)
print(y.shape)
print(blocks.shape)

(30, 9, 1500)
(30,)
(30,)


In [43]:
from ssvep.preprocessing import bandpass, notch, crop

X_filt = notch(X, fs=config.BENCHMARK_FS)
print(X_filt.shape)
X_filt = bandpass(X_filt, fs=config.BENCHMARK_FS, low=config.BANDPASS[0], high=config.BANDPASS[1])
print(X_filt.shape)
X_filt = X_filt[:, None]
print(X_filt.shape)
start = config.BENCHMARK_ONSET + config.BENCHMARK_LATENCY

(30, 9, 1500)
(30, 9, 1500)
(30, 1, 9, 1500)


In [44]:
from ssvep.cca import CCA

window_lengths = [0.5, 1.0, 1.5, 2.0]
score_dict = {}
for window_length in window_lengths:
    print(f"Window length: {window_length}")
    X_new = crop(X_filt, fs=config.BENCHMARK_FS, start=start, duration=window_length)
    print(X_new.shape)

    cca = CCA(freqs=config.TARGET_FREQS, fs=config.BENCHMARK_FS)
    cca.fit(X_new, y)
    print(cca.predict(X_new))
    score = cca.score(X_new, y)
    print(f"score: {score}")
    score_dict[window_length] = score

Window length: 0.5
(30, 1, 9, 125)
[0 1 2 2 4 2 1 0 1 0 0 2 1 2 4 2 0 2 3 4 0 1 2 3 0 0 1 2 1 4]
score: 0.6
Window length: 1.0
(30, 1, 9, 250)
[0 1 2 3 2 0 1 2 0 3 0 1 2 2 0 0 1 2 3 4 0 1 2 3 4 0 1 2 3 4]
score: 0.8333333333333334
Window length: 1.5
(30, 1, 9, 375)
[0 1 2 3 2 0 1 2 1 4 0 1 2 2 4 0 1 2 3 4 0 1 2 2 3 0 1 2 3 4]
score: 0.8333333333333334
Window length: 2.0
(30, 1, 9, 500)
[0 1 2 3 4 0 1 2 3 4 0 1 2 3 3 0 1 2 3 2 0 1 2 3 4 0 1 2 3 4]
score: 0.9333333333333333


In [45]:
from ssvep.evaluation import itr

for window_length, score in score_dict.items():
    print(f"Window length: {window_length}, ITR: {itr(5, score, window_length)}")

Window length: 0.5, ITR: 66.1173000519232
Window length: 1.0, ITR: 80.31434039434049
Window length: 1.5, ITR: 53.542893596226996
Window length: 2.0, ITR: 55.05706279597823
